# A/B Test Analysis: Ad Campaign vs. PSA Control

**Business question:** Does showing users the actual ad campaign increase conversion compared with showing a Public Service Announcement (PSA) instead?

**Dataset:** `marketing_AB.csv` — 588,101 users with treatment group, conversion outcome, ad exposure, and day/hour fields.

### What this notebook covers
1. Data validation
2. Sample Ratio Mismatch (SRM) check
3. Primary hypothesis test
4. Confidence interval and lift
5. Statistical vs. practical significance
6. Day/hour segment checks
7. Exposure check
8. Final decision

> **Note:** `most_ads_day` and `most_ads_hour` are used only for descriptive segmentation. They were not separately randomized.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

pd.set_option('display.float_format', lambda x: f'{x:.5f}')

df = pd.read_csv('marketing_AB.csv', index_col=0)
df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
df['converted'] = df['converted'].astype(bool)

df.head()

## 1. Data Validation

Before running the test, I checked the dataset shape, missing values, group counts, and group proportions.

In [ ]:
print('Shape:', df.shape)
print('\nMissing values:')
print(df.isna().sum())

print('\nGroup counts:')
print(df['test_group'].value_counts())

print('\nGroup proportions:')
print(df['test_group'].value_counts(normalize=True))

The allocation is intentionally unequal: about **96%** of users are in `ad` and **4%** are in `psa`. That is why a simple 50/50 split check would not be appropriate here.

I use the documented 96/4 allocation for the SRM check below.

## 2. Sample Ratio Mismatch (SRM) Check

An SRM check asks whether the observed group sizes are consistent with the intended experiment allocation. A mismatch can point to a problem with assignment or logging.

Here, the intended allocation is **96% ad / 4% psa**, so I use a chi-square goodness-of-fit test.

In [ ]:
observed = df['test_group'].value_counts()[['ad', 'psa']].values
total = observed.sum()
expected_ratio = np.array([0.96, 0.04])
expected = total * expected_ratio

chi2, p_srm = stats.chisquare(observed, expected)

print(f'Observed counts: {observed}')
print(f'Expected counts: {expected.round(2)}')
print(f'Chi-square statistic: {chi2:.6f}')
print(f'SRM p-value: {p_srm:.4f}')

if p_srm < 0.01:
    print('SRM detected — investigate assignment/logging before relying on the experiment result.')
else:
    print('No SRM detected — observed allocation is consistent with the intended 96/4 design.')

**Result:** No SRM was detected (p = 0.9998). The observed allocation is consistent with the intended 96/4 design.

This does **not** prove that randomization was perfect; it only shows that the observed group split does not provide evidence of a sample-ratio problem.

## 3. Primary Hypothesis Test

The primary metric is conversion rate, which is binary: converted or not converted.

**H₀:** The conversion rates are equal between `ad` and `psa`.

**H₁:** The conversion rates are different between `ad` and `psa`.

**Significance level:** α = 0.05

I use a **two-proportion z-test** because the outcome is a binary conversion rate and the two groups are being compared.

In [ ]:
conv_ad = df.loc[df['test_group'] == 'ad', 'converted']
conv_psa = df.loc[df['test_group'] == 'psa', 'converted']

n_ad, n_psa = len(conv_ad), len(conv_psa)
x_ad, x_psa = conv_ad.sum(), conv_psa.sum()

rate_ad = x_ad / n_ad
rate_psa = x_psa / n_psa

count = np.array([x_ad, x_psa])
nobs = np.array([n_ad, n_psa])

z_stat, p_value = proportions_ztest(count, nobs, alternative='two-sided')

print(f'Ad:  {x_ad:,} conversions / {n_ad:,} users = {rate_ad:.4%}')
print(f'PSA: {x_psa:,} conversions / {n_psa:,} users = {rate_psa:.4%}')
print(f'Absolute difference: {(rate_ad - rate_psa):.4%}')
print(f'z-statistic: {z_stat:.3f}')
print(f'p-value: {p_value:.2e}')

**Result:** p ≈ 1.7 × 10⁻¹³, which is far below 0.05. The observed conversion rates are therefore statistically different, with the `ad` group having the higher observed conversion rate.

## 4. Confidence Interval and Lift

The p-value tells us about the evidence against the null hypothesis, but it does not tell us how large the difference is. So I also calculate the absolute lift and its 95% confidence interval.

In [ ]:
diff = rate_ad - rate_psa
se_diff = np.sqrt(
    rate_ad * (1 - rate_ad) / n_ad +
    rate_psa * (1 - rate_psa) / n_psa
)

ci_low = diff - 1.96 * se_diff
ci_high = diff + 1.96 * se_diff
relative_lift = diff / rate_psa

print(f'Absolute lift: {diff:.4%} ({diff * 100:.2f} percentage points)')
print(f'95% CI: [{ci_low:.4%}, {ci_high:.4%}]')
print(f'Relative lift: {relative_lift:.1%}')

**Result:** The absolute lift is about **+0.77 percentage points**, with a 95% CI of roughly **+0.60 to +0.94 percentage points**. The relative lift is about **+43.1%** over the PSA conversion rate.

Because the confidence interval is entirely above zero, it agrees with the hypothesis-test result.

## 5. Effect Size with Confidence Interval

This chart gives a quick visual of the estimated lift and the uncertainty around it.

In [ ]:
lift_pp = diff * 100
ci_low_pp = ci_low * 100
ci_high_pp = ci_high * 100

plt.figure(figsize=(7, 4))
plt.errorbar(
    x=[lift_pp],
    y=[0],
    xerr=[[lift_pp - ci_low_pp], [ci_high_pp - lift_pp]],
    fmt='o',
    capsize=6
)
plt.axvline(0, linestyle='--')
plt.yticks([])
plt.xlabel('Absolute conversion lift (percentage points)')
plt.title('Ad vs. PSA: Estimated Conversion Lift with 95% CI')
plt.tight_layout()
plt.show()

## 6. Statistical vs. Practical Significance

**Statistical significance:** Yes. The p-value is far below 0.05 and the confidence interval does not cross zero.

**Practical significance:** The observed +0.77 percentage-point lift is large enough to be worth discussing from a business point of view, especially because the experiment contains more than 588K users.

However, the dataset does not contain campaign cost, revenue per conversion, or downstream customer value. So I cannot calculate ROI from this analysis alone. The business value should be checked using those numbers before a permanent rollout.

## 7. Segment Analysis

I also checked the observed lift by day of week and by broad hour-of-day buckets. These are descriptive checks, not separate randomized experiments.

In [ ]:
day_pivot = (
    df.groupby(['test_group', 'most_ads_day'])['converted']
      .mean()
      .unstack()
      .T
)
day_pivot['lift'] = day_pivot['ad'] - day_pivot['psa']
day_pivot

In [ ]:
df['hour_bucket'] = pd.cut(
    df['most_ads_hour'],
    bins=[-1, 5, 11, 17, 23],
    labels=['Night (0-5)', 'Morning (6-11)', 'Afternoon (12-17)', 'Evening (18-23)']
)

hour_pivot = (
    df.groupby(['test_group', 'hour_bucket'], observed=True)['converted']
      .mean()
      .unstack()
      .T
)
hour_pivot['lift'] = hour_pivot['ad'] - hour_pivot['psa']
hour_pivot

The `ad` group has a higher observed conversion rate across the tested day and hour segments. The largest relative difference appears in the night bucket.

I would treat this as a useful consistency check rather than proof of a separate causal effect within each segment, because users were not re-randomized inside these day/hour cuts.

In [ ]:
ad_exposure = (
    df.groupby('test_group')['total_ads']
      .describe()[['count', 'mean', '50%', 'max']]
)
ad_exposure

**Exposure check:** the median recorded ad exposure is similar between the groups (13 for `ad` vs. 12 for `psa`), and the mean exposure is also close.

This is useful descriptive information, but `total_ads` is a post-treatment variable. Because of that, I would not use exposure as an independent causal adjustment for the main treatment effect.

## 8. Conclusion

The experiment shows a higher observed conversion rate for the ad group than for the PSA group:

- **Ad conversion:** 2.55%
- **PSA conversion:** 1.79%
- **Absolute lift:** +0.77 percentage points
- **95% CI:** approximately +0.60 to +0.94 percentage points
- **Relative lift:** +43.1%
- **p-value:** approximately 1.7 × 10⁻¹³
- **SRM:** no mismatch detected

### Decision

The results support using the ad experience over the PSA control for the primary conversion metric tested. Before making the change permanent, I would also check the economics of the additional conversions and validate the result with a follow-up experiment.

The detailed stakeholder-facing write-up is in `AB_Test_Decision_Memo.docx`.

## 9. Limitations

- The control group is much smaller because of the 96/4 allocation.
- Day/hour results are descriptive because those dimensions were not separately randomized.
- Exposure is a post-treatment variable, so it should not be interpreted as a clean causal adjustment.
- The dataset does not include financial information, so ROI cannot be calculated here.
- A single experiment gives useful evidence, but a follow-up test would help confirm that the observed effect is repeatable.